# Compute daily mean wind speed

To align with DCPP model data, we will compute daily wind speed from daily mean u and v components.

- ERA5
- BARRA-R2
- BARRA-C2 later - not a reanalysis as no DA

In [1]:
from dask_setup import setup_dask_client 
    
client, cluster, dask_tmp = setup_dask_client(mode="interactive", workload_type="cpu")

INFO     [client] Starting Dask client setup (workload_type=cpu | environment=jupyter)
INFO     [resources] Resources detected via PBS (total_cores=104 | total_mem_gib=496.0)


INFO     [client] Temp/spill dir: /jobfs/181663634.gadi-pbs/dask-2276912
INFO     [client] Workers: 104 | threads/worker: 1 | processes: True
INFO     [client] Mem: total ~496.0 GiB | usable ~446.0 GiB | per-worker ~4.3 GiB
INFO     [client] Compression: spill=auto | comm=False
INFO     [client] Dask client ready


[setup_dask_client] Configuration summary
temp/spill dir: /jobfs/181663634.gadi-pbs/dask-2276912
Workers: 104 | threads/worker: 1 | processes: True
Memory: total ~496.0 GiB | usable ~446.0 GiB | per-worker ~4.3 GiB
Compression: spill=auto | comm=False


In [2]:
import xarray as xr

import glob

In [3]:
%cd /g/data/w42/dr6273/work/wind_drought/
# import functions as fn

%load_ext autoreload
%autoreload 2

/g/data/w42/dr6273/work/wind_drought


In [4]:
BARRA_R2_PATH = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/" # daily mean (?)

BARRA_R2_1HR_PATH = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/" # daily mean

WRITE_PATH = "/g/data/ng72/dr6273/work/projects/wind_drought/data/BARRA-R2/"

# Compute wind speed by year

In [5]:
def barraR2_preprocess(ds):
    """
    Preprocess function for open_mfdataset.
    Selects Australian region, computes daily mean and renames coords.
    """
    ds = ds.sel(
        lon=slice(REGION[0], REGION[1]),
        lat=slice(REGION[3], REGION[2])
    )
    # ds["time"] = ds["time"].dt.floor("D")
    # ds = ds.chunk({"time": -1, "lat": -1, "lon": -1})
    # ds = ds.astype("float32")
    return ds

In [6]:
# def open_barra(files):
#     """
#     Open multiple files and preprocess to region.
    
#     fp: str, path to file. Should not include files, only the path to dir.
#     lat_slice, lon_slice: slice of lat/lon to subset
#     lat_name, lon_name: names of lat/lon coords.
#     """
#     # def preprocess(ds):
#     #     # ds = ds.rename({lat_name: "lat"})
#     #     # ds = ds.rename({lon_name: "lon"})
#     #     ds = ds.sel(lon=lon_slice, lat=lat_slice)
#     #     # ds["time"] = ds["time"].dt.round(freq="20min")
#     #     return ds.astype("float32")
    
#     ds = xr.open_mfdataset(
#         files,
#         preprocess=barraR2_preprocess,
#         chunks={"time": "300MB"},# "lat": -1, "lon": -1},
#         concat_dim="time",
#         combine="nested",
#         compat="override",
#         coords="minimal",
#         data_vars="minimal",
#         # decode_times=True
#         # parallel=True
#     )
#     return ds

In [7]:
# def get_filename(variable, frequency, year, month):
#     """
#     Return BARRA-R2 AUS-11 historical data filepaths
    
#     variable: str, variable name e.g. ua100m
#     frequency: str, timestep e.g. 20min, 1hr
#     year: str
#     month: str, month in format '01', '02', ..., '12'
#     """
#     file_template = variable + "_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_" + frequency + "_"
#     return barra_path + frequency + "/" + variable + "/latest/" + file_template + year + month + "-" + year + month + ".nc"

In [8]:
import numpy as np

In [9]:
YEARS = range(1979, 1981)

In [10]:
import src.read as read
import src.regions as regions
import src.wind as wind

In [11]:
REGION = regions.get_Aus_boundary()

In [22]:
# SCRATCH_PATH = '/scratch/w42/dr6273/BARRA-R2/'

In [13]:
chunks = {"time": 24 * 7 * 2, "lat": -1, "lon": -1}

In [18]:
for i, year in enumerate(YEARS[:]):
    start = year * 100 + 1
    end = year * 100 + 12

    u100_files = read.barra_get_files(BARRA_R2_1HR_PATH, 'ua100m', start=start, end=end)
    v100_files = read.barra_get_files(BARRA_R2_1HR_PATH, 'va100m', start=start, end=end)

    u100 = read.open_files(u100_files, preprocess=barraR2_preprocess)
    v100 = read.open_files(v100_files, preprocess=barraR2_preprocess)

    u100_chunked = u100.chunk(chunks)
    v100_chunked = v100.chunk(chunks)

    u100_chunked = u100_chunked["ua100m"]
    v100_chunked = v100_chunked["va100m"]

    capacity_factor = wind.capacity_factor(np.hypot(u100_chunked, v100_chunked))

    capacity_factor = capacity_factor.coarsen(time=24).mean() # Hourly to daily mean
    capacity_factor["time"] = capacity_factor["time"].dt.floor("D")
    
    capacity_factor = capacity_factor.chunk({"time": -1, "lat": -1, "lon": -1})
    
    capacity_factor = capacity_factor.to_dataset(name="capacity_factor")
    
    capacity_factor_encoding = read.specify_encoding(capacity_factor)

    if i == 0:
        read.write_zarr(
            capacity_factor,
            WRITE_PATH,
            'barra-r2_capacity_factor_100m_'+str(YEARS[0]) + '-' + str(YEARS[-1]) + '.zarr',
            capacity_factor_encoding,
        )
    else:
        read.write_zarr(
            capacity_factor,
            WRITE_PATH,
            'barra-r2_capacity_factor_100m_'+str(YEARS[0]) + '-' + str(YEARS[-1]) + '.zarr',
            None,
            mode="a",
            append_dim="time"
        )

In [19]:
ds = xr.open_zarr(
    WRITE_PATH + 'barra-r2_capacity_factor_100m_'+str(YEARS[0]) + '-' + str(YEARS[-1]) + '.zarr',
    consolidated=False
)

In [23]:
ds.capacity_factor

<xarray.DataArray 'capacity_factor' (time: 731, lat: 319, lon: 409)> Size: 381MB
dask.array<open_dataset-capacity_factor, shape=(731, 319, 409), dtype=float32, chunksize=(365, 319, 409), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 6kB 1979-01-01 1979-01-02 ... 1980-12-31
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    height   float64 8B ...
    crs      int32 4B ...
Attributes:
    long_name:      Eastward Wind at 100m
    standard_name:  eastward_wind
    units:          m s-1
    cell_methods:   model_level_number: point area: interpolation (method: bi...
    grid_mapping:   crs

In [36]:
capacity_factor['capacity_factor']#.chunk({"time": 24 * 7 * 2, "lat": -1, "lon": -1})

<xarray.DataArray 'capacity_factor' (time: 365, lat: 319, lon: 409)> Size: 381MB
dask.array<rechunk-merge, shape=(365, 319, 409), dtype=float64, chunksize=(365, 319, 409), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 3kB 1979-01-01 1979-01-02 ... 1979-12-31
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    height   float64 8B 100.0
    crs      int32 4B 0
Attributes:
    long_name:      Eastward Wind at 100m
    standard_name:  eastward_wind
    units:          m s-1
    cell_methods:   model_level_number: point area: interpolation (method: bi...
    grid_mapping:   crs

In [ ]:
u100

Process Dask Worker process (from Nanny):
2026-10-20 12:59:17,966 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2026-10-20 12:59:17,967 - distributed.nanny - ERROR - Worker process died unexpectedly
Traceback (most recent call last):
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/asyncio/base_events.py", line 691, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/nanny.py", line 985, in run
    await worker.finished()
  File "/g/data/xp65/public/apps/

In [16]:
start = YEARS[0] * 100 + 1
end = YEARS[-1] * 100 + 12

In [17]:
u100_files = read.barra_get_files(BARRA_R2_1HR_PATH, 'ua100m', start=start, end=end)
v100_files = read.barra_get_files(BARRA_R2_1HR_PATH, 'va100m', start=start, end=end)

In [18]:
u100_files[-10:]

[PosixPath('/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/ua100m/latest/ua100m_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_202503-202503.nc'),
 PosixPath('/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/ua100m/latest/ua100m_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_202504-202504.nc'),
 PosixPath('/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/ua100m/latest/ua100m_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_202505-202505.nc'),
 PosixPath('/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/ua100m/latest/ua100m_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_202506-202506.nc'),
 PosixPath('/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/ua100m/latest/ua100m_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_1hr_202507-202507.nc'),
 PosixPath('/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5

In [19]:
u100 = read.open_files(u100_files, preprocess=barraR2_preprocess)
v100 = read.open_files(v100_files, preprocess=barraR2_preprocess)

In [27]:
u100["ua100m"].chunk({"time": 24 * 7 * 104, "lat": 60, "lon": 40})

<xarray.DataArray 'ua100m' (time: 412008, lat: 319, lon: 409)> Size: 430GB
dask.array<rechunk-p2p, shape=(412008, 319, 409), dtype=float64, chunksize=(17472, 60, 40), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 3MB 1979-01-01 ... 2025-12-31T23:00:00
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    height   float64 8B 100.0
    crs      int32 4B 0
Attributes:
    long_name:      Eastward Wind at 100m
    standard_name:  eastward_wind
    units:          m s-1
    cell_methods:   model_level_number: point area: interpolation (method: bi...
    grid_mapping:   crs

In [28]:
# chunks = {"time": 24 * 7 * 104, "lat": 60, "lon": 40}

In [29]:
chunks = {"time": 24 * 7 * 104, "lat": 60, "lon": 40}
u100_chunked = u100.chunk(chunks)
v100_chunked = v100.chunk(chunks)

In [30]:
u100_chunked = u100_chunked["ua100m"]
v100_chunked = v100_chunked["va100m"]

In [31]:
u100_chunked

<xarray.DataArray 'ua100m' (time: 412008, lat: 319, lon: 409)> Size: 430GB
dask.array<rechunk-p2p, shape=(412008, 319, 409), dtype=float64, chunksize=(17472, 60, 40), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 3MB 1979-01-01 ... 2025-12-31T23:00:00
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    height   float64 8B 100.0
    crs      int32 4B 0
Attributes:
    long_name:      Eastward Wind at 100m
    standard_name:  eastward_wind
    units:          m s-1
    cell_methods:   model_level_number: point area: interpolation (method: bi...
    grid_mapping:   crs

In [32]:
# wind_speed = np.hypot(u100_chunked, v100_chunked) #np.sqrt(u100_chunked ** 2 + v100_chunked ** 2)
# wind_speed = wind_speed.chunk({"time": -1, "lat": -1, "lon": 5})

# wind = wind_speed.to_dataset(name="wind_speed")

In [38]:
capacity_factor = wind.capacity_factor(np.hypot(u100_chunked, v100_chunked))

capacity_factor = capacity_factor.coarsen(time=24).mean() # Hourly to daily mean
capacity_factor["time"] = capacity_factor["time"].dt.floor("D")

capacity_factor = capacity_factor.chunk({"time": -1, "lat": 160, "lon": 16})

capacity_factor = capacity_factor.to_dataset(name="capacity_factor")

capacity_factor_encoding = read.specify_encoding(capacity_factor)

In [39]:
capacity_factor['capacity_factor']

<xarray.DataArray 'capacity_factor' (time: 17167, lat: 319, lon: 409)> Size: 18GB
dask.array<rechunk-merge, shape=(17167, 319, 409), dtype=float64, chunksize=(17167, 160, 16), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 137kB 1979-01-01 1979-01-02 ... 2025-12-31
  * lat      (lat) float64 3kB -44.99 -44.88 -44.77 ... -10.23 -10.12 -10.01
  * lon      (lon) float64 3kB 110.0 110.2 110.3 110.4 ... 154.7 154.8 154.9
    height   float64 8B 100.0
    crs      int32 4B 0
Attributes:
    long_name:      Eastward Wind at 100m
    standard_name:  eastward_wind
    units:          m s-1
    cell_methods:   model_level_number: point area: interpolation (method: bi...
    grid_mapping:   crs

In [ ]:
read.write_zarr(
    capacity_factor,
    WRITE_PATH,
    'barra-r2_capacity_factor_100m_'+str(YEARS[0]) + '-' + str(YEARS[-1]) + '.zarr',
    capacity_factor_encoding
)

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 245.25 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [20]:
# from zarr.codecs import BloscCodec
wind_encoding = read.specify_encoding(wind)

In [22]:
# wind.wind_speed.isel(time=0).plot()

In [31]:
read.write_zarr(
    wind,
    WRITE_PATH,
    'barra-r2_wind_speed_100m_1979-2025.zarr',
    wind_encoding
)

2026-10-19 13:24:55,960 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('getitem-open_dataset-concatenate-5d4291eb362e9b89048eaeade935dac5', 937, 0, 0))" coro=<Worker.execute() done, defined at /g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/worker_state_machine.py:3608>> ended with CancelledError
2026-10-19 13:24:55,960 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('getitem-open_dataset-concatenate-5d4291eb362e9b89048eaeade935dac5', 400, 0, 0))" coro=<Worker.execute() done, defined at /g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/worker_state_machine.py:3608>> ended with CancelledError
2026-10-19 13:24:55,961 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute(('getitem-open_dataset-concatenate-5d4291eb362e9b89048eaeade935dac5', 398, 0,

KeyboardInterrupt: 

2026-10-19 13:24:57,995 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/asyncio/base_events.py", line 691, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/nanny.py", line 985, in run
    await worker.finished()
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/site-packages/distributed/core.py", line 494, in finished
    await self._event_finished.wait()
  File "/g/data/xp65/public/apps/med_conda/envs/analysis3-26.09/lib/python3.12/asyncio/locks.py", l

In [15]:
def barraR2_windspeed(u_name, v_name, ws_name, years, write_path):
    """
    Compute wind speed from daily mean u and v winds.

    u_name, v_name: str, names of u and v variables in BARRA-R2
    ws_name: str, desired wind speed variable name
    years: range, years to process
    write_path: str, path to write directory
    """
    da_list = []
    for year in years:
        if year % 5 == 0:
            print(year)
            
        # monthly_arrays = []
        # for month in month_str[:]:
            
        u = open_barra(
            sorted(glob.glob(BARRA_R2_PATH + u_name + "/latest/*_" + str(year) + "*.nc")),
        ).astype("float32")
        
        v = open_barra(
            sorted(glob.glob(BARRA_R2_PATH + v_name + "/latest/*_" + str(year) + "*.nc")),
        ).astype("float32")
        
        ws = fn.windspeed(
            u.rename({u_name: ws_name}),
            v.rename({v_name: ws_name})
        )[ws_name]

        ws["time"] = ws["time"].dt.floor("D")

        # Time chunk size of 1 as it is the greatest common divisor of
        #. the different time steps associated with each year (365 and 366)
        # ws = ws.chunk({"time": 1, "lat": -1, "lon": -1})
        
        ws = ws.to_dataset(name=ws_name)
        
        encoding = {
            ws_name: {"dtype": "float32"}
        }

        ws.to_netcdf(
            write_path + ws_name + "_BARRA-R2_daily_Aus_"+str(year)+".nc",
            mode="w"
        )
        
        # if year == 1979:
        #     ws.to_zarr(
        #         write_path + ws_name + "_BARRA-R2_daily_Aus.zarr",
        #         mode="w",
        #         encoding=encoding,
        #         consolidated=True,
        #         zarr_format=2, # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
        #     )
        # else:
        #     ws.to_zarr(
        #         write_path + ws_name + "_BARRA-R2_daily_Aus.zarr",
        #         mode="a",
        #         append_dim="time",
        #         consolidated=True,
        #         zarr_format=2, # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
        #     )

        # return ws

        # da_list.append(ws)
        
    # ws = xr.concat(da_list, dim="time")

    # encoding = {
    #     ws_name: {"dtype": "float32"}
    # }

    # ws = ws.chunk({"time": 550, "lat": -1, "lon": -1})
    # ws = ws.to_dataset(name=ws_name)
    # ws.to_zarr(
    #     write_path + ws_name + "_BARRA-R2_daily_Aus.zarr",
    #     consolidated=True,
    #     mode="w",
    #     zarr_format=2, # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
    #     encoding=encoding
    # )
    # # return ws

In [13]:
# barraR2_windspeed("ua100m", "va100m", "ws100m", YEARS[3:], WRITE_PATH)

In [77]:
BARRA_R2_PATH

'/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/'

In [18]:
YEARS[10:]

range(1989, 2025)

In [19]:
barraR2_windspeed("uas", "vas", "wss", YEARS[10:], WRITE_PATH)

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 26.11 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


1990


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 26.11 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 25.48 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


1995
2000
2005


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 13.25 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


2010
2015
2020


# Close cluster

In [21]:
client.close()
cluster.close()